In [1]:
!pip install mediapipe opencv-python torch torchvision
!pip install face_recognition

  Using cached torchvision-0.25.0-cp312-cp312-win_amd64.whl.metadata (5.4 kB)
   ---------------------------------------- 0.0/113.8 MB ? eta -:--:--
    --------------------------------------- 1.8/113.8 MB 10.1 MB/s eta 0:00:12
   - -------------------------------------- 3.4/113.8 MB 8.8 MB/s eta 0:00:13
   - -------------------------------------- 5.5/113.8 MB 9.3 MB/s eta 0:00:12
   -- ------------------------------------- 7.3/113.8 MB 9.4 MB/s eta 0:00:12
   --- ------------------------------------ 9.2/113.8 MB 9.4 MB/s eta 0:00:12
   --- ------------------------------------ 11.3/113.8 MB 9.4 MB/s eta 0:00:11
   ---- ----------------------------------- 13.6/113.8 MB 9.6 MB/s eta 0:00:11
   ----- ---------------------------------- 15.7/113.8 MB 9.6 MB/s eta 0:00:11
   ------ --------------------------------- 17.6/113.8 MB 9.7 MB/s eta 0:00:10
   ------ --------------------------------- 19.7/113.8 MB 9.7 MB/s eta 0:00:10
   ------- -------------------------------- 21.0/113.8 MB 9.7 MB/

  error: subprocess-exited-with-error
  
  exit code: 1
  
  [41 lines of output]
  running bdist_wheel
  running build
  running build_ext
  
  
                     CMake is not installed on your system!
  
      Or it is possible some broken copy of cmake is installed on your system.
      It is unfortunately very common for python package managers to include
      broken copies of cmake.  So if the error above this refers to some file
      path to a cmake file inside a python or anaconda or miniconda path then you
      should delete that broken copy of cmake from your computer.
  
      Instead, please get an official copy of cmake from one of these known good
      sources of an official cmake:
          - cmake.org (this is how windows users should get cmake)
          - apt install cmake (for Ubuntu or Debian based systems)
          - yum install cmake (for Redhat or CenOS based systems)
  
      On a linux machine you can run `which cmake` to see what cmake you are
      act

In [ ]:
import os
import cv2
import random
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
import face_recognition
from sklearn.metrics import confusion_matrix, f1_score
from tqdm import tqdm
import re
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [ ]:
# =====================================================================
# 📂 1. 데이터 리스트 준비 (기존 코드와 동일)
# =====================================================================
def get_combined_data_list(daisee_csv, daisee_dir, yawdd_m_dir, yawdd_f_dir, is_train=True):
    data_list = []

    if os.path.exists(daisee_csv):
        df = pd.read_csv(daisee_csv)
        df_01 = df[df['Engagement'].isin([0, 1])]
        rand_num = 330 if is_train else 210
        df_3 = df[df['Engagement'] == 3]
        df_3 = df_3.sample(n=min(rand_num, len(df_3)))
        df_sampled = pd.concat([df_01, df_3]).reset_index(drop=True)

        for _, row in df_sampled.iterrows():
            clip_id = str(row['ClipID']).replace('.avi', '').replace('.mp4', '')
            label = 1 if int(row['Engagement']) >= 2 else 0
            subject_id = clip_id[:6]
            possible_paths = [
                os.path.join(daisee_dir, f"{clip_id}.avi"),
                os.path.join(daisee_dir, f"{clip_id}.mp4"),
                os.path.join(daisee_dir, subject_id, clip_id, f"{clip_id}.avi"),
                os.path.join(daisee_dir, subject_id, clip_id, f"{clip_id}.mp4")
            ]
            video_path = next((p for p in possible_paths if os.path.exists(p)), None)
            if video_path: data_list.append((video_path, label))

    if os.path.exists(yawdd_f_dir):
        for root, _, files in os.walk(yawdd_f_dir):
            for file in files:
                if file.lower().endswith(('.avi', '.mp4')):
                    video_path = os.path.join(root, file)
                    filename_lower = file.lower()
                    num_search = re.search(r'\d+', filename_lower)
                    if num_search:
                        number = int(num_search.group())
                        if 'yawning' in filename_lower and number > 30 and not is_train: label = 0
                        elif 'yawning' in filename_lower and number <= 30 and is_train: label = 0
                        else: continue
                        data_list.append((video_path, label))

    if os.path.exists(yawdd_m_dir):
        for root, _, files in os.walk(yawdd_m_dir):
            for file in files:
                if file.lower().endswith(('.avi', '.mp4')):
                    video_path = os.path.join(root, file)
                    filename_lower = file.lower()
                    num_search = re.search(r'\d+', filename_lower)
                    if num_search:
                        number = int(num_search.group())
                        if 'yawning' in filename_lower and number > 30 and not is_train: label = 0
                        elif 'yawning' in filename_lower and number <= 30 and is_train: label = 0
                        else: continue
                        data_list.append((video_path, label))

    if is_train: random.shuffle(data_list)

    focus_count = sum(1 for _, label in data_list if label == 1)
    unfocus_count = sum(1 for _, label in data_list if label == 0)
    mode = "Train" if is_train else "Val"
    print(f"📊 [{mode}] 데이터 밸런스 확인\n   - 집중(1): {focus_count}개\n   - 비집중(0): {unfocus_count}개\n   - 총합: {len(data_list)}개\n")
    return data_list

In [ ]:
# =====================================================================
# 🚀 2. Dataset & Collate Fn (프레임을 한 장씩 낱개로 추출!)
# =====================================================================
class FrameFaceDataset(Dataset):
    def __init__(self, data_list, is_train=False):
        self.data_list = data_list
        self.is_train = is_train
        self.stride = 4

        self.base_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.Grayscale(num_output_channels=3),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        # self.augment = transforms.Compose([
        #     transforms.RandomHorizontalFlip(p=0.5),
        #     transforms.RandomRotation(degrees=15),
        #     transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
        #     transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))
        # ])

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        video_path, label = self.data_list[idx]
        frames = []
        labels = []
        last_box = None

        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            return [torch.zeros((3, 224, 224))], [label]

        frame_idx = 0

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            # 지정된 간격(stride)마다 프레임 추출
            if frame_idx % self.stride == 0:
                rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                h, w = frame.shape[:2]

                if last_box is None:
                    face_locations = face_recognition.face_locations(rgb_frame, model="hog")
                    if face_locations:
                        t, r, b, l = face_locations[0]
                        pad_x = int((r - l) * 0.4)
                        pad_y = int((b - t) * 0.4)
                        last_box = (max(0, t - pad_y), min(h, b + pad_y), max(0, l - pad_x), min(w, r + pad_x))
                    else:
                        last_box = (int(h*0.2), int(h*0.8), int(w*0.2), int(w*0.8))

                face_crop = rgb_frame[last_box[0]:last_box[1], last_box[2]:last_box[3]]
                face_pil = transforms.ToPILImage()(face_crop)

                # 16프레임 묶음 없이, 프레임 1장마다 개별적으로 증강 적용
                # if self.is_train:
                #     face_pil = self.augment(face_pil)

                # [3, 224, 224] 형태의 단일 이미지 텐서를 리스트에 추가
                frames.append(self.base_transform(face_pil))
                labels.append(label)

            frame_idx += 1
        cap.release()

        # 영상에서 하나도 못 건졌을 때 예외 처리
        if len(frames) == 0:
            frames.append(torch.zeros((3, 224, 224)))
            labels.append(label)

        # 묶음(chunks)이 아닌 개별 프레임 리스트를 반환!
        return frames, labels

In [ ]:
def separate_collate_fn(batch):
    # 영상을 하나로 뭉치지 않고, 각 영상별로 텐서를 만들어 리스트로 묶어 반환합니다.
    # 결과: [ (영상1_프레임들, 라벨들), (영상2_프레임들, 라벨들), ... ]
    processed_batch = []
    for frames_list, labels_list in batch:
        frames_tensor = torch.stack(frames_list) # [T, 3, 224, 224] (T는 해당 영상의 추출 프레임 수)
        labels_tensor = torch.tensor(labels_list, dtype=torch.float32) # [T]
        processed_batch.append((frames_tensor, labels_tensor))
    return processed_batch

In [ ]:
# =====================================================================
# 🧠 3. 표준 2D 모델 (MobileNetV2)
# =====================================================================
class FrameMobileNetV2(nn.Module):
    def __init__(self, num_classes=2):
        super(FrameMobileNetV2, self).__init__()
        # 3D/비디오 구조를 버리고 표준 2D MobileNetV2 사용
        self.backbone = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        num_ftrs = self.backbone.classifier[1].in_features

        # 마지막 분류기 층 교체
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(num_ftrs, num_classes)
        )

    def forward(self, x):
        # x 입력 형태: [Batch, Channel, Height, Width]
        return self.backbone(x)

In [ ]:
# =====================================================================
# 🌟 Focal Loss 클래스 추가
# =====================================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        return focal_loss.sum()

In [ ]:
# =====================================================================
# ⚙️ 4. 메인 실행부
# =====================================================================
# 🚨 경로 확인!
DAISEE_TRAIN_CSV = "/content/drive/MyDrive/DeepLearning/dataset/DAiSEE/Labels/TrainLabels.csv"
DAISEE_VAL_CSV   = "/content/drive/MyDrive/DeepLearning/dataset/DAiSEE/Labels/ValidationLabels.csv"
DAISEE_TRAIN_DIR = "/content/drive/MyDrive/DeepLearning/dataset/DAiSEE/DataSet/Train"
DAISEE_VAL_DIR   = "/content/drive/MyDrive/DeepLearning/dataset/DAiSEE/DataSet/Validation"
YAWDD_M_DIR      = "/content/drive/MyDrive/DeepLearning/dataset/Yawdd/Mirror/Mirror/Male_mirror_Avi_Videos"
YAWDD_F_DIR      = "/content/drive/MyDrive/DeepLearning/dataset/Yawdd/Mirror/Mirror/Female_mirror"
SAVE_PATH        = "/content/drive/MyDrive/DeepLearning/dataset/out/best_focus_model_frame.pth"

print("🔍 데이터 스캔 및 병합을 시작합니다...")
train_list = get_combined_data_list(DAISEE_TRAIN_CSV, DAISEE_TRAIN_DIR, YAWDD_M_DIR, YAWDD_F_DIR, is_train=True)
val_list   = get_combined_data_list(DAISEE_VAL_CSV, DAISEE_VAL_DIR, YAWDD_M_DIR, YAWDD_F_DIR, is_train=False)

# 배치 사이즈는 2~4 정도가 적당합니다. (영상 2개에서 나온 프레임들을 한 번에 처리)
train_loader = DataLoader(FrameFaceDataset(train_list, is_train=True), batch_size=2, shuffle=True, num_workers=0, collate_fn=separate_collate_fn)
val_loader   = DataLoader(FrameFaceDataset(val_list, is_train=False), batch_size=2, shuffle=False, num_workers=0, collate_fn=separate_collate_fn)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FrameMobileNetV2(num_classes=2).to(device)

# 모델 깊은 층 봉인 해제 (fc 대신 classifier로 이름 변경됨)
for name, param in model.named_parameters():
    if any(x in name for x in ["features.14", "features.15", "features.16", "features.17", "features.18", "classifier"]):
        param.requires_grad = True
    else:
        param.requires_grad = False

criterion = FocalLoss(gamma=2.0)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5, weight_decay=5e-2)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

num_epochs = 100
best_val_loss = float('inf')
patience_counter = 0

# 메트릭 저장 리스트
train_losses = []
val_losses = []
train_accs = []
val_accs = []
f1_scores = []
epochs_list = []

for epoch in range(num_epochs):
    print(f"\n==============================================")
    print(f" 🚀 Epoch {epoch+1}/{num_epochs} 시작! (영상별 개별 프레임 학습)")
    print(f"==============================================")

    model.train()
    running_loss, correct, total = 0.0, 0, 0

    train_pbar = tqdm(train_loader, desc="[Train]")
    for batch in train_pbar:
        # ★ 핵심: 뭉쳐진 배치가 아니라, batch 안의 영상들을 '각각' 하나씩 꺼내서 학습합니다.
        for inputs, labels in batch:
            # inputs 형태: [T, 3, 224, 224] (특정 영상 하나의 프레임 T개)
            inputs, labels = inputs.to(device), labels.to(device).long()

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step() # 영상 1개 분량을 보고 가중치 업데이트!

            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += torch.sum(preds == labels.data)
            total += inputs.size(0)

        train_pbar.set_postfix({'Loss': f'{loss.item():.4f}'})

    # === Validation (검증) ===
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="[Val]"):
            for inputs, labels in batch: # 검증도 각각 꺼내서 평가
                inputs, labels = inputs.to(device), labels.to(device).long()
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                val_correct += torch.sum(preds == labels.data)
                val_total += inputs.size(0)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

    epoch_val_loss = val_loss / val_total

    print(f"\n[Train] Loss: {running_loss/total:.4f} | Acc: {correct.double()/total:.4f}")
    print(f"[Val]   Loss: {epoch_val_loss:.4f} | Acc: {val_correct.double()/val_total:.4f} | F1: {f1_score(all_labels, all_preds):.4f}")
    print("Confusion Matrix:\n", confusion_matrix(all_labels, all_preds))

    # 메트릭 저장
    train_losses.append(running_loss / total)
    val_losses.append(epoch_val_loss)
    train_accs.append(correct.double() / total)
    val_accs.append(val_correct.double() / val_total)
    f1_scores.append(f1_score(all_labels, all_preds))
    epochs_list.append(epoch + 1)

    scheduler.step(epoch_val_loss)

    if epoch_val_loss < best_val_loss:
        print(f"🎉 [신기록 달성!] Val Loss가 {best_val_loss:.4f} ➡️ {epoch_val_loss:.4f} 로 감소했습니다!")
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"💾 최적의 모델이 저장되었습니다: {SAVE_PATH}")
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"⚠️ 최고 기록 갱신 실패... (Early Stopping 카운트: {patience_counter}/8)")

    if patience_counter >= 8:
        print("\n🛑 Early stopping triggered. 학습을 조기 종료합니다.")
        break

In [ ]:
# 🚨 경로 세팅
DAISEE_TRAIN_CSV = "/content/drive/MyDrive/DeepLearning/dataset/DAiSEE/Labels/TrainLabels.csv"
DAISEE_VAL_CSV   = "/content/drive/MyDrive/DeepLearning/dataset/DAiSEE/Labels/ValidationLabels.csv"
DAISEE_TRAIN_DIR = "/content/drive/MyDrive/DeepLearning/dataset/DAiSEE/DataSet/Train"
DAISEE_VAL_DIR   = "/content/drive/MyDrive/DeepLearning/dataset/DAiSEE/DataSet/Validation"
YAWDD_M_DIR      = "/content/drive/MyDrive/DeepLearning/dataset/Yawdd/Mirror/Mirror/Male_mirror_Avi_Videos"
YAWDD_F_DIR      = "/content/drive/MyDrive/DeepLearning/dataset/Yawdd/Mirror/Mirror/Female_mirror"

# 📌 이전에 저장해둔 모델 경로 (불러올 뇌)
LOAD_MODEL_PATH  = "/content/drive/MyDrive/DeepLearning/dataset/out/best_focus_model_frame.pth"
# 📌 새롭게 갱신해서 저장할 경로 (덮어쓰기 방지를 위해 이름을 살짝 바꿉니다)
SAVE_MODEL_PATH  = "/content/drive/MyDrive/DeepLearning/dataset/out/best_focus_model_frame_resumed.pth"

print("🔍 데이터 스캔 및 병합을 시작합니다...")
train_list = get_combined_data_list(DAISEE_TRAIN_CSV, DAISEE_TRAIN_DIR, YAWDD_M_DIR, YAWDD_F_DIR, is_train=True)
val_list   = get_combined_data_list(DAISEE_VAL_CSV, DAISEE_VAL_DIR, YAWDD_M_DIR, YAWDD_F_DIR, is_train=False)

train_loader = DataLoader(FrameFaceDataset(train_list, is_train=True), batch_size=2, shuffle=True, num_workers=0, collate_fn=separate_collate_fn)
val_loader   = DataLoader(FrameFaceDataset(val_list, is_train=False), batch_size=2, shuffle=False, num_workers=0, collate_fn=separate_collate_fn)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FrameMobileNetV2(num_classes=2).to(device)

# ==========================================================
# 🧠 [핵심] 기존 모델 가중치 불러오기 (이어서 학습)
# ==========================================================
if os.path.exists(LOAD_MODEL_PATH):
    print(f"\n✅ 저장된 모델을 발견했습니다! 가중치를 불러옵니다: {LOAD_MODEL_PATH}")
    model.load_state_dict(torch.load(LOAD_MODEL_PATH, map_location=device))
else:
    print(f"\n⚠️ 저장된 모델이 없습니다. 처음부터 학습을 시작합니다.")

# 모델 깊은 층 봉인 해제 (Fine-Tuning)
for name, param in model.named_parameters():
    if any(x in name for x in ["features.14", "features.15", "features.16", "features.17", "features.18", "classifier"]):
        param.requires_grad = True
    else:
        param.requires_grad = False

criterion = FocalLoss(gamma=2.0)

# 이미 학습된 뇌가 놀라지 않도록, 학습률(Learning Rate)을 5e-5에서 1e-5로 확 낮춰서 조심스럽게 다듬습니다.
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5, weight_decay=5e-2)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

# 이어서 학습할 추가 에포크 수 설정
num_epochs = 15
best_val_loss = float('inf')
patience_counter = 0

# 메트릭 저장 리스트
train_losses = []
val_losses = []
train_accs = []
val_accs = []
f1_scores = []
epochs_list = []

for epoch in range(num_epochs):
    print(f"\n==============================================")
    print(f" 🚀 Resume Epoch {epoch+1}/{num_epochs} 시작! (비집중 편향 교정 중...) ")
    print(f"==============================================")

    model.train()
    running_loss, correct, total = 0.0, 0, 0

    train_pbar = tqdm(train_loader, desc="[Train]")
    for batch in train_pbar:
        for inputs, labels in batch:
            inputs, labels = inputs.to(device), labels.to(device).long()

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += torch.sum(preds == labels.data)
            total += inputs.size(0)

        train_pbar.set_postfix({'Loss': f'{loss.item():.4f}'})

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="[Val]"):
            for inputs, labels in batch:
                inputs, labels = inputs.to(device), labels.to(device).long()
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                val_correct += torch.sum(preds == labels.data)
                val_total += inputs.size(0)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

    epoch_val_loss = val_loss / val_total

    print(f"\n[Train] Loss: {running_loss/total:.4f} | Acc: {correct.double()/total:.4f}")
    print(f"[Val]   Loss: {epoch_val_loss:.4f} | Acc: {val_correct.double()/val_total:.4f} | F1: {f1_score(all_labels, all_preds):.4f}")
    print("Confusion Matrix:\n", confusion_matrix(all_labels, all_preds))

    # 메트릭 저장
    train_losses.append(running_loss / total)
    val_losses.append(epoch_val_loss)
    train_accs.append(correct.double() / total)
    val_accs.append(val_correct.double() / val_total)
    f1_scores.append(f1_score(all_labels, all_preds))
    epochs_list.append(epoch + 1)

    scheduler.step(epoch_val_loss)

    if epoch_val_loss < best_val_loss:
        print(f"🎉 [신기록 달성!] Val Loss가 {best_val_loss:.4f} ➡️ {epoch_val_loss:.4f} 로 감소했습니다!")
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), SAVE_MODEL_PATH)
        print(f"💾 교정된 모델이 새롭게 저장되었습니다: {SAVE_MODEL_PATH}")
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"⚠️ 최고 기록 갱신 실패... (Early Stopping 카운트: {patience_counter}/5)")

    if patience_counter >= 7:
        print("\n🛑 Early stopping triggered. 편향 교정 학습을 조기 종료합니다.")
        break

In [ ]:
# =====================================================================
# 📊 학습 결과 그래프 시각화
# =====================================================================
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 8))

# Loss 그래프
plt.subplot(2, 2, 1)
plt.plot(epochs_list, train_losses, label='Train Loss', marker='o')
plt.plot(epochs_list, val_losses, label='Val Loss', marker='o')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Accuracy 그래프
plt.subplot(2, 2, 2)
plt.plot(epochs_list, train_accs, label='Train Accuracy', marker='o')
plt.plot(epochs_list, val_accs, label='Val Accuracy', marker='o')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# F1 Score 그래프
plt.subplot(2, 2, 3)
plt.plot(epochs_list, f1_scores, label='F1 Score', marker='o', color='green')
plt.title('F1 Score over Epochs')
plt.xlabel('Epoch')
plt.ylabel('F1 Score')
plt.legend()
plt.grid(True)

# Val Loss와 Val Acc 비교
plt.subplot(2, 2, 4)
plt.plot(epochs_list, val_losses, label='Val Loss', marker='o', color='red')
plt.plot(epochs_list, val_accs, label='Val Accuracy', marker='o', color='blue')
plt.title('Validation Loss and Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Value')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()